<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 10B: *Fire Damage Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_damage = pd.read_csv('../data/processed/X_damage.csv')
y_damage = pd.read_csv('../data/processed/y_damage.csv').squeeze()  # Load as Series
details_damage = pd.read_csv('../data/processed/details_damage.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/damage_best_strategy.csv')

with open('../data/processed/model_parameters_damage.json', 'r') as f:
    model_parameters = json.load(f)

with open('../data/processed/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_damage,y_damage], axis=1)
subset = subset_df(reform, 'Target_Damage', 500)

y = subset['Target_Damage']
X = subset.drop(columns='Target_Damage')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build Final tuned models
damage_xgb = xgb.XGBClassifier(**XGB_parameters)
damage_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 20,
 'min_samples_split': 5,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 5,
 'n_estimators': 200,
 'max_depth': 6,
 'learning_rate': 0.3,
 'verbosity': 0}

In [7]:
models = {
    "XGB": damage_xgb, 
    "RF": damage_rf
}

## SHAP

In [8]:
xgb_top = get_shap(damage_xgb, X_xgb, y_xgb)
rf_top = get_shap(damage_rf, X_rf, y_rf,)

 99%|===================| 2480/2500 [01:19<00:00]        

In [9]:
weather_features = []

drop = ['fire_count','fire_count 3 Day Sum','fire_count 7 Day Sum','fire_count 30 Day Sum',
       'road_density_x_forest_percent','power_line_density_x_log_total_housing']
weather_sets = ['Water Demand','Water Supply','Water Supply Indexes','Fire Danger','Interactions','Wind Slope',
                              'Lag 3 Day Features','Lag 7 Day Features','Lag 30 Day Features']

for weather_set in weather_sets:
    for feature in feature_sets[weather_set]:
        if feature not in drop:
            weather_features.append(feature)

In [10]:
top_xgb_weather = xgb_top.loc[xgb_top['Feature'].isin(weather_features)]
top_xgb_weather[:5]

,Feature,SHAP Importance
5,SPEI 90-Day,0.415770
6,Specific Humidity 30 Day Mean,0.399149
8,Palmer Drought Severity Index,0.371612
13,Solar Radiation 7 Day Sum,0.305643
15,Actual Evapotranspiration 7 Day Sum,0.293510


In [11]:
top_rf_weather = rf_top.loc[rf_top['Feature'].isin(weather_features)]
top_rf_weather[:5]

,Feature,SHAP Importance
0,Vapor Pressure Deficit 30 Day Mean,0.016514
7,Daily Maximum Air Temperature 30 Day Mean,0.009856
9,Solar Radiation 7 Day Sum,0.009512
10,Specific Humidity 30 Day Mean,0.009155
13,Solar Radiation 3 Day Sum,0.007648


In [12]:
common_ranked = (
    rf_top[:20][['Feature', 'SHAP Importance']]
    .rename(columns={'SHAP Importance': 'RF_Importance'})
    .merge(
        xgb_top[:20][['Feature', 'SHAP Importance']]
        .rename(columns={'SHAP Importance': 'XGB_Importance'}),
        on='Feature',
        how='inner'
    )
    .sort_values('XGB_Importance', ascending=False)
    .reset_index(drop=True)
)

In [13]:
xgb_top[:10]

,Feature,SHAP Importance
0,intermix_zone,0.727017
1,avg_dist_to_all_reservoirs_same_day,0.664152
2,elevation_range,0.571089
3,dominant_section_percent,0.536020
4,fire_count 30 Day Sum,0.474153
5,SPEI 90-Day,0.415770
6,Specific Humidity 30 Day Mean,0.399149
7,elevation_mean,0.392921
8,Palmer Drought Severity Index,0.371612
9,median_income,0.370476


In [14]:
rf_top[:10]

,Feature,SHAP Importance
0,Vapor Pressure Deficit 30 Day Mean,0.016514
1,intermix_zone,0.014537
2,influence_zone,0.012090
3,slope_mean,0.011916
4,fire_count 30 Day Sum,0.010570
5,avg_dist_to_all_reservoirs_same_day,0.010360
6,median_income,0.009993
7,Daily Maximum Air Temperature 30 Day Mean,0.009856
8,elevation_mean,0.009534
9,Solar Radiation 7 Day Sum,0.009512


In [15]:
common_ranked

,Feature,RF_Importance,XGB_Importance
0,intermix_zone,0.014537,0.727017
1,avg_dist_to_all_reservoirs_same_day,0.010360,0.664152
2,dominant_section_percent,0.008960,0.536020
3,fire_count 30 Day Sum,0.010570,0.474153
4,SPEI 90-Day,0.007583,0.415770
5,Specific Humidity 30 Day Mean,0.009155,0.399149
6,elevation_mean,0.009534,0.392921
7,Palmer Drought Severity Index,0.006860,0.371612
8,median_income,0.009993,0.370476
9,slope_mean,0.011916,0.312412


## Set Ablation

In [16]:
Ablation_single_column = ablation(models, X, y, feature_sets, best_strategy, single_set = False)

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Infrastructure
Testing RF: Infrastructure
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons
Testing XGB: Spatial
Testing RF: Spatial
Testing XGB: Lag 3 Day Features
Testing RF: Lag 3 Day Features
Testing XGB: Lag 7 Day Features
Testing RF: Lag 7 Day Features
Testing XGB: Lag 30 Day Features
Testing RF: Lag 30 Day Features


In [17]:
# Assume your table is in a DataFrame called df
# Filter out the full runs
full_model = Ablation_single_column[Ablation_single_column['Test Set'] == 'Full'][['Model', 'Macro F1']].rename(columns={'Macro F1': 'Full Macro F1'})

# Merge back to the main dataframe on 'Model'
pivot_ablation = Ablation_single_column.merge(full_model, on='Model', how='left')

# Drop columns you don’t need for now
pivot_ablation = pivot_ablation.drop(columns=['Weighted F1', 'High Risk Recall'])

largest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] > .2]
smallest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] < 0]

In [18]:
largest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
4,Fire Danger,XGB,0.911305,0.245769,0.913550
14,Coded Ecoregions,XGB,0.911348,0.240999,0.913550
15,Coded Seasons,XGB,0.911378,0.237774,0.913550
16,Spatial,XGB,0.905653,0.864370,0.913550
21,Water Demand,RF,0.927884,0.851641,0.935854
22,Water Supply,RF,0.928323,0.804770,0.935854
23,Water Supply Indexes,RF,0.901712,3.648202,0.935854
24,Fire Danger,RF,0.929938,0.632209,0.935854
25,Social,RF,0.930003,0.625239,0.935854
26,Infrastructure,RF,0.926121,1.040074,0.935854


In [19]:
smallest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
1,Water Demand,XGB,0.915798,-0.246057,0.91355
2,Water Supply,XGB,0.917437,-0.425470,0.91355
3,Water Supply Indexes,XGB,0.913665,-0.012632,0.91355
5,Social,XGB,0.915609,-0.225372,0.91355
6,Infrastructure,XGB,0.919768,-0.680623,0.91355
7,Elevation,XGB,0.930274,-1.830717,0.91355
8,WUI,XGB,0.915596,-0.223938,0.91355
9,Ecoregion,XGB,0.919847,-0.689338,0.91355
10,Land Cover,XGB,0.917697,-0.453919,0.91355
11,Interactions,XGB,0.917937,-0.480251,0.91355


In [20]:
Ablation_single_column

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.913162,0.913550,0.989583,0.000000
1,Water Demand,XGB,0.915145,0.915798,0.989583,-0.246057
2,Water Supply,XGB,0.917064,0.917437,0.989583,-0.425470
3,Water Supply Indexes,XGB,0.913243,0.913665,0.989583,-0.012632
4,Fire Danger,XGB,0.911076,0.911305,0.989583,0.245769
5,Social,XGB,0.914668,0.915609,0.989583,-0.225372
6,Infrastructure,XGB,0.918686,0.919768,0.989583,-0.680623
7,Elevation,XGB,0.928904,0.930274,0.989583,-1.830717
8,WUI,XGB,0.915365,0.915596,0.989583,-0.223938
9,Ecoregion,XGB,0.919000,0.919847,0.989583,-0.689338
